# Cascade — 축 분해 구조를 주장 단위로 검증한다

3-class 확률은 **두 축으로 정확히 쪼개진다.**

$$p=\big[(1-t)\,r,\;\;(1-t)(1-r),\;\;t\big],\qquad
\text{총 ll}=\underbrace{\text{tie축}}_{t\ \text{만의 함수}}+\underbrace{\text{승부축}}_{r\ \text{만의 함수}}$$

이 노트북은 그 구조 위에서 **주장 7개를 순서대로 세우고 하나씩 판정**한다.
각 절은 `주장 -> 근거표 -> 판정` 세 덩어리로만 되어 있고, 맨 끝 §9 가 판정 전부를 한 화면에 다시 찍는다.

| # | 주장 | 비용 | 판정에 쓰는 것 |
|---|---|---|---|
| 1 | 축 분해는 무손실이다 — `총 ll = tie축 + 승부축` 은 항등식이다 | 0 | 수치 검산 |
| 2 | 두 축은 서로 간섭하지 않는다 — 축별로 따로 보정해도 전체 최적이다 | 0 | 축별 적합 vs 2차원 격자 공동최적화 |
| 3 | **각 전문가에게 자기 축만 맡기고 온도 2개면 된다** (방법 1) | 0 | 학습 없는 조립 vs 학습된 결합기 |
| 4 | 학습된 joint 부스터가 그보다 낫다 | 부스터 2회 | `J`, `J-rep` vs 방법 1 |
| 5 | 헤드를 둘로 나눠 학습하면 joint 를 이긴다 | 부스터 8회 | `C0`~`C6` vs `J-rep` |
| 6 | 그래서 배포에 계층 구조를 넣어야 한다 | 0 | 채택 규칙(문턱 0.0005 + CI) |
| 7 | 이기든 지든 **어느 층에서** 갈렸는지 설명된다 | 0 | 클래스·집중도·보정·층화 차이표 |

**신호 정의.** softmax 는 상수 덧셈에 불변이라 `z_tie` 단독은 의미가 없다. 전문가마다 두 축의 좌표만 뽑는다.

| 전문가 | tie 신호 `s` | 승부 신호 `d` |
|---|---|---|
| ABE (3-class) | `z_tie − logaddexp(z_a, z_b)` = `logit p(tie)` | `z_a − z_b` = `logit p(A│비-tie)` |
| TB (2-class) | `z_tie − z_notie` | **없음** — swap 불변이라 방향 정보가 구조적으로 0 |
| Arith (raw margin) | 〃 | 〃 |

**swap 규율.** `s` 는 swap 불변(+1), `d` 는 부호반전(−1). tie 헤드는 불변 열만 받으므로 **정의상 대칭**이고
증강·TTA 가 불필요하다. 승부 쪽은 `(x·부호, 1−y)` 증강 + `r=(f(x)+1−f(x·부호))/2` TTA 로 반대칭을 강제한다.
두 계약 모두 아래에서 기계 정밀도로 검산한다.

**실행.** 위에서부터 그냥 돌리면 된다. 무거운 절(주장 4·5)은 `HEAVY=False` 로 건너뛸 수 있고,
그 경우에도 주장 1~3 과 §9 요약은 그대로 나온다.

### §1 설정

In [ ]:
import os, json, glob, time, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 42
np.random.seed(SEED)

LLM        = Path('.')
ABE_ROOT   = LLM / 'V5/aws/outputs/best_checkpoints'
TB_ROOT    = LLM / 'V6/pod/outputs/best_checkpoints'
ARITH_LOGP = LLM / 'Arith/outputs/arith_experts.npz'      # 구 규약: 확률평균 후 log
ARITH_RAW  = LLM / 'Arith/outputs/arith_experts_raw.npz'  # 신 규약: raw margin 평균 (Boost/arith_raw.py)
ORIGIN     = LLM / 'Arith/origin'
OUT        = LLM / 'Boost'
MODEL_DIR  = OUT / 'model_cascade'
OOF_PATH   = OUT / 'cascade_oof.npz'

# ── 손잡이 ──
HEAVY        = True          # False 면 주장 4·5(부스터 학습)를 건너뛴다. 주장 1~3 은 학습이 없다
HEAD_BOOST   = 'cat'         # 두 헤드에 쓸 부스터. 'lgbm' 이 3~4배 빠르다
ABE_ENSEMBLE = 'all'         # None(단일 최선) | 'all'(전 체크포인트 로짓평균, 현행 채택) | [step,...]
TB_ENSEMBLE  = None          # TB 는 앙상블이 손해라 단일 유지 (tie축 0.5785 -> 0.5800)
CKPT_SELECT  = 'valid'       # 단일 선택 시의 기준 (§1.6 프로토콜)
ARITH_SRC    = 'raw'         # 'raw': ABE/TB 와 같은 raw 로짓 규약 | 'logp': 구 규약
ARITH_HEAD   = 'mean'        # 'mean': 부스터 3종 평균 -> 신호 1쌍 | 'all': 부스터별 3쌍
WIN_TRAIN    = 'soft'        # AB 헤드 학습행. 'soft': tie 행 반쪽 가중 두 벌 | 'nontie': 비-tie 만
N_SPLITS     = 5
INNER_FRAC   = 0.10
SHOW_CKPT    = False         # 체크포인트별 점수표를 찍을지 (기본 요약만)

# 배포 구성은 CV argmin 으로 고르지 않는다 — 보고 지표로 고르면 낙관이 섞인다.
DEFAULT_CFG  = 'C0 요청안'   # 사전 지정안
DEPLOY_CFG   = None          # None: 규칙대로 | 문자열: 강제
ADOPT_THRESH = 0.0005        # 채택 문턱 (§1.6 규율)

ARITH_NPZ, ARI_PRE = ((ARITH_RAW, 'raw') if ARITH_SRC == 'raw' and ARITH_RAW.exists()
                      else (ARITH_LOGP, 'logp'))
for p in (ABE_ROOT, TB_ROOT, ARITH_NPZ, ORIGIN / 'valid.csv', ORIGIN / 'holdout.csv'):
    assert p.exists(), f'경로 없음: {p}'
assert HEAD_BOOST in ('lgbm', 'xgb', 'cat')
assert ARITH_HEAD in ('mean', 'all') and WIN_TRAIN in ('soft', 'nontie')
os.makedirs(MODEL_DIR, exist_ok=True)

print(f'부스터 {HEAD_BOOST} | ABE 앙상블 {ABE_ENSEMBLE} | TB 앙상블 {TB_ENSEMBLE} | '
      f'Arith {ARI_PRE}/{ARITH_HEAD} | AB 학습행 {WIN_TRAIN} | {N_SPLITS}-fold, ES {INNER_FRAC:.0%}'
      + ('' if HEAVY else '  | HEAVY=False (주장 4·5 생략)'))
if ARI_PRE != ARITH_SRC:
    print('  ! arith_experts_raw.npz 없음 -> 구 규약(logp)으로 내려감. Boost/arith_raw.py 먼저 실행')

### §2 출력 유틸 — 한글 폭을 세는 표 출력기 + 판정 기록

In [ ]:
# 파이썬 문자열 포매팅은 글자 수를 세지만 터미널은 폭을 센다 — 한글은 2칸이라 %-16s 가 어긋난다.
# 아래 표 출력기가 이 노트북의 모든 표를 담당한다 (정렬이 깨지면 결과를 읽을 수 없다).
def _w(s):
    return sum(2 if unicodedata.east_asian_width(ch) in 'WF' else 1 for ch in str(s))

def _pad(s, n, right=False):
    s = str(s)
    fill = ' ' * max(0, n - _w(s))
    return fill + s if right else s + fill

def table(headers, rows, align=None, sep='─'):
    """headers: [str], rows: [[cell]]. align: 'l'/'r' 문자열 (기본 첫 열 l, 나머지 r)."""
    rows = [[('' if c is None else c) for c in r] for r in rows]
    align = align or ('l' + 'r' * (len(headers) - 1))
    wd = [max(_w(h), *(_w(r[i]) for r in rows)) if rows else _w(h)
          for i, h in enumerate(headers)]
    line = lambda cells: '  '.join(_pad(c, wd[i], align[i] == 'r') for i, c in enumerate(cells))
    print(line(headers))
    print(sep * _w(line(headers)))
    for r in rows:
        print(line(r))

# ── 판정 기록 ── 각 절이 여기에 결과를 남기고 §9 가 한 화면에 다시 찍는다
VERDICTS = []

def claim(n, text):
    bar = '━' * 96
    print(f'{bar}\n[주장 {n}] {text}\n{bar}')

def verdict(n, text, ok, detail):
    mark = '성립' if ok else ('기각' if ok is False else '보류')
    VERDICTS.append({'#': n, '주장': text, '판정': mark, '근거': detail})
    print(f'\n  >> 판정: {mark} — {detail}\n')

f4 = lambda v: f'{v:.4f}'
fD = lambda v: f'{v:+.4f}'

### §3 데이터 + 전문가 로짓

In [ ]:
def labels_of(d):
    y = np.select([d['winner_model_a'] == 1, d['winner_model_b'] == 1, d['winner_tie'] == 1],
                  [0, 1, 2], default=-1).astype(np.int64)
    assert (y != -1).all(), 'invalid labels!'
    return y

va, ho = pd.read_csv(ORIGIN / 'valid.csv'), pd.read_csv(ORIGIN / 'holdout.csv')
assert len(va) == 5000 and len(ho) == 5748, f'분할 크기 {len(va)}/{len(ho)}'
IDS = np.concatenate([va['id'].to_numpy(), ho['id'].to_numpy()])
Y = np.concatenate([labels_of(va), labels_of(ho)])
HALF, N = len(va), len(IDS)
assert len(np.unique(IDS)) == N, 'id 중복'
prior = np.bincount(Y, minlength=3) / N

# Boost_Ens.ipynb 와 동일한 fold 배정 (StratifiedKFold 는 y 와 행 수만 본다) -> 결과가 직접 비교된다
FOLD = np.zeros(N, int)
for _k, (_, _te) in enumerate(StratifiedKFold(N_SPLITS, shuffle=True,
                                              random_state=SEED).split(np.zeros((N, 1)), Y)):
    FOLD[_te] = _k

import torch
import torch.nn as nn

def fit_temperature(logits, labels):
    z = torch.tensor(logits, dtype=torch.float32)
    y = torch.tensor(labels, dtype=torch.long)
    lg = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([lg], lr=0.1, max_iter=100)
    def cl():
        opt.zero_grad()
        loss = nn.functional.cross_entropy(z / lg.exp(), y)
        loss.backward()
        return loss
    opt.step(cl)
    return float(lg.exp().detach())

def load_one(path):
    z = np.load(path)
    L = np.vstack([(z['valid_fwd'] + z['valid_swp']) / 2,
                   (z['hold_fwd'] + z['hold_swp']) / 2]).astype(np.float64)
    assert (np.concatenate([z['valid_ids'], z['hold_ids']]) == IDS).all(), f'id 불일치: {path}'
    assert (np.concatenate([z['valid_labels'], z['hold_labels']]) == Y).all(), f'라벨 불일치: {path}'
    return L

def score_logits(tag, L):
    v, h, vl, hl = L[:HALF], L[HALF:], Y[:HALF], Y[HALF:]
    if tag == 'abe':
        T = fit_temperature(v, vl)
        sm = lambda x: torch.softmax(torch.tensor(x / T, dtype=torch.float32), 1).numpy()
        return log_loss(vl, sm(v), labels=[0, 1, 2]), log_loss(hl, sm(h), labels=[0, 1, 2]), T
    T = fit_temperature(v, (vl == 2).astype(np.int64))
    sm = lambda x: torch.softmax(torch.tensor(x / T, dtype=torch.float32), 1).numpy()[:, 1]
    b = lambda p, l: log_loss((l == 2).astype(int), np.c_[1 - p, p], labels=[0, 1])
    return b(sm(v), vl), b(sm(h), hl), T

def pick_and_load(tag, root, n_class, ensemble):
    paths = {int(p.rsplit('-', 1)[1]): Path(p) for p in glob.glob(str(root / 'checkpoint-*'))}
    byid = {s: load_one(paths[s] / 'logits_export.npz') for s in sorted(paths)}
    rows = [(s, *score_logits(tag, byid[s])) for s in sorted(paths)]
    best = min(rows, key=lambda r: r[1] if CKPT_SELECT == 'valid' else r[2])
    steps = ([best[0]] if ensemble is None
             else sorted(paths) if ensemble == 'all' else sorted(int(s) for s in ensemble))
    if SHOW_CKPT:
        table(['step', 'valid', 'hold', 'T', ''],
              [[s, f4(lv), f4(lh), f'{T:.3f}',
                ('단독최선 ' if s == best[0] else '') + ('[앙상블]' if len(steps) > 1 and s in steps else '')]
               for s, lv, lh, T in rows], align='lrrrl')
    L = byid[steps[0]] if len(steps) == 1 else np.mean([byid[s] for s in steps], axis=0)
    lv, lh, T = score_logits(tag, L)
    assert L.shape == (N, n_class) and np.isfinite(L).all(), f'{tag} 형상/유한성 위반 {L.shape}'
    kind = f'{len(steps)}벌 로짓평균' if len(steps) > 1 else f'단일 ckpt-{steps[0]}'
    print(f'  {tag.upper():<5} {kind:<16} valid {lv:.4f}  hold {lh:.4f}  T {T:.3f}'
          + (f'   (단독최선 대비 {lv - best[1]:+.4f}/{lh - best[2]:+.4f})' if len(steps) > 1 else ''))
    return steps, T, L

print(f'stage2 {N:,}행 (valid {HALF:,} + hold {N - HALF:,}) | 사전분포 a/b/tie {np.round(prior, 4)} | '
      f'fold {np.bincount(FOLD)}')
ABE_STEPS, ABE_T, L_ABE = pick_and_load('abe', ABE_ROOT, 3, ABE_ENSEMBLE)
TB_STEPS, TB_T, L_TB = pick_and_load('tb', TB_ROOT, 2, TB_ENSEMBLE)
assert L_TB.shape[1] == 2, 'TB 는 2-class 다 — [1,0,2] 재정렬 금지'

za = np.load(ARITH_NPZ)
assert (np.concatenate([za['ids_valid'], za['ids_hold']]) == IDS).all(), 'Arith id 불일치'
assert (np.concatenate([za['y_valid'], za['y_hold']]) == Y).all(), 'Arith 라벨 불일치'
ARITH_MODELS = ('lgbm', 'xgb', 'cat')
_blocks = [np.vstack([za[f'{ARI_PRE}_valid__{m}'], za[f'{ARI_PRE}_hold__{m}']]).astype(np.float64)
           for m in ARITH_MODELS]
ARI_BLOCKS = [np.mean(_blocks, axis=0)] if ARITH_HEAD == 'mean' else _blocks
ARI_NAMES = ['ari'] if ARITH_HEAD == 'mean' else list(ARITH_MODELS)
assert all(np.isfinite(b).all() for b in ARI_BLOCKS), 'Arith 비유한값'
print(f'  ARITH {ARI_PRE:<5} {ARITH_HEAD:<16} 블록 {len(ARI_BLOCKS)}개 {ARI_NAMES}')

### §4 신호 정의 + swap 계약 + 지표

In [ ]:
def sd_of(blk):
    # 3-class 블록 -> (s, d). 자유도 2 라 softmax 출력에 대해 무손실이다.
    return blk[:, 2] - np.logaddexp(blk[:, 0], blk[:, 1]), blk[:, 0] - blk[:, 1]

COL = {}                                   # 이름 -> (값, swap 부호)
_s_tb = L_TB[:, 1] - L_TB[:, 0]
COL['s_tb']      = (_s_tb, +1.0)                      # TB tie 신호
COL['notie_raw'] = (L_TB[:, 0], +1.0)                 # TB 의 raw non-tie 로짓 (크기 조절용)
COL['notie_s']   = (-_s_tb, +1.0)                     # 정규화판 logit p(비-tie)
_s, _d = sd_of(L_ABE)
COL['s_abe'], COL['d_abe'], COL['|d_abe|'] = (_s, +1.0), (_d, -1.0), (np.abs(_d), +1.0)
for _nm, _b in zip(ARI_NAMES, ARI_BLOCKS):
    _s, _d = sd_of(_b)
    COL[f's_{_nm}'], COL[f'd_{_nm}'], COL[f'|d_{_nm}|'] = (_s, +1.0), (_d, -1.0), (np.abs(_d), +1.0)

S_ARI  = [f's_{n}' for n in ARI_NAMES]
D_ARI  = [f'd_{n}' for n in ARI_NAMES]
AD_ARI = [f'|d_{n}|' for n in ARI_NAMES]

def build(names, kind):
    X = np.column_stack([COL[n][0] for n in names])
    sg = np.array([COL[n][1] for n in names])
    if kind == 'tie':
        bad = [n for n, s in zip(names, sg) if s < 0]
        assert not bad, f'tie 헤드는 swap 불변 열만 받는다 — 위반: {bad}'
    else:
        assert (sg < 0).any(), f'AB 헤드에 방향 열이 없다: {names}'
    return X, sg

table(['신호', 'swap', '평균', 'std', '역할'],
      [[k, f'{int(s):+d}', f'{v.mean():+.3f}', f'{v.std():.3f}',
        'tie축' if s > 0 and not k.startswith('|d') else ('승부축' if s < 0 else 'tie축(마진)')]
       for k, (v, s) in COL.items()])

# ── 지표 ── 축 분해는 임의의 확률벡터에 성립하는 항등식이라 매번 검산한다
EPS = 1e-15

def per_row_ll(p, y):
    p = np.clip(np.asarray(p, np.float64), EPS, 1.0)
    p = p / p.sum(1, keepdims=True)
    return -np.log(p[np.arange(len(y)), y])

def axis_report(p, y):
    p = np.clip(np.asarray(p, np.float64), EPS, 1.0)
    p = p / p.sum(1, keepdims=True)
    total = per_row_ll(p, y).mean()
    it = y == 2
    t = np.clip(p[:, 2], EPS, 1 - EPS)
    tie_axis = float(-np.where(it, np.log(t), np.log(1 - t)).mean())
    nz = ~it
    rA = np.clip(p[nz, 0] / (p[nz, 0] + p[nz, 1]), EPS, 1 - EPS)
    win_axis = float(-np.where(y[nz] == 0, np.log(rA), np.log(1 - rA)).sum() / len(y))
    assert abs(total - tie_axis - win_axis) < 1e-6, '축 분해 검산 실패'
    return {'총 ll': float(total), 'tie축': tie_axis, '승부축': win_axis,
            'tie AUC': float(roc_auc_score(it.astype(int), t)),
            '승부 AUC': float(roc_auc_score((y[nz] == 0).astype(int), rA)),
            'acc%': float((p.argmax(1) == y).mean() * 100)}

def paired_ci(ll_a, ll_b, n_boot=2000, seed=SEED):
    d = np.asarray(ll_a) - np.asarray(ll_b)
    rng = np.random.default_rng(seed)
    bs = d[rng.integers(0, len(d), (n_boot, len(d)))].mean(1)
    return float(d.mean()), float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975))

def ci_str(m, lo, hi, thresh=ADOPT_THRESH):
    sig = '유의' if (lo > 0 or hi < 0) else '노이즈'
    return f'{m:+.4f} [{lo:+.4f},{hi:+.4f}] {sig}' + (' ★' if sig == '유의' and abs(m) >= thresh else '')

NOINFO_TIE = float(-(prior[2] * np.log(prior[2]) + (1 - prior[2]) * np.log(1 - prior[2])))
NOINFO_WIN = float(np.log(2) * (1 - prior[2]))
print(f'\n무정보 기준선  tie축 {NOINFO_TIE:.4f} + 승부축 {NOINFO_WIN:.4f} = {NOINFO_TIE + NOINFO_WIN:.4f}'
      f'   | 사전분포 {-np.log(prior[Y]).mean():.4f} | V4 blend 0.9844 / 게이트 0.9812')

RES = {}          # 이름 -> {'oof', 'mean', 'std', ...}. 모든 절이 여기에 쌓인다

## 주장 1 — 축 분해는 무손실이다

$$\text{총 ll}=-\tfrac1N\!\sum\log p_y
=\underbrace{-\tfrac1N\!\sum\big[y_{tie}\log t+(1-y_{tie})\log(1-t)\big]}_{\text{tie축}}
+\underbrace{-\tfrac1N\!\sum_{y\neq tie}\log r_y}_{\text{승부축}}$$

**임의의** 확률벡터에 성립하는 항등식이고, tie축은 `t` 에만 승부축은 `r` 에만 의존한다.
성립하면 "축을 나눠 다루는 것" 자체는 공짜다 — 이후 모든 주장이 이 위에 선다.

In [ ]:
claim(1, '총 ll = tie축 + 승부축 은 임의의 확률벡터에 성립하는 항등식이다')

_rng = np.random.default_rng(SEED)
_cases = {
    '난수(디리클레)': _rng.dirichlet([1, 1, 1], N),
    '난수(극단)': _rng.dirichlet([0.05, 0.05, 0.05], N),
    '사전분포 상수': np.tile(prior, (N, 1)),
    'ABE softmax': (lambda z: z / z.sum(1, keepdims=True))(np.exp(L_ABE - L_ABE.max(1, keepdims=True))),
}
_rows, _emax = [], 0.0
for _nm, _p in _cases.items():
    _r = axis_report(_p, Y)
    _e = abs(_r['총 ll'] - _r['tie축'] - _r['승부축'])
    _emax = max(_emax, _e)
    _rows.append([_nm, f4(_r['총 ll']), f4(_r['tie축']), f4(_r['승부축']), f'{_e:.2e}'])
table(['확률벡터', '총 ll', 'tie축', '승부축', '|잔차|'], _rows)

# 축을 따로 건드리면 다른 축이 정말 안 움직이는지 — 항등식의 실질적 내용
_p = _cases['ABE softmax']
_t, _r0 = _p[:, 2], _p[:, 0] / np.clip(_p[:, 0] + _p[:, 1], EPS, None)
_join = lambda t, r: np.column_stack([(1 - t) * r, (1 - t) * (1 - r), t])
_a = axis_report(_join(np.clip(_t * 0.8, EPS, 1 - EPS), _r0), Y)          # tie축만 변형
_b = axis_report(_join(_t, np.clip(_r0 * 0.8, EPS, 1 - EPS)), Y)          # 승부축만 변형
_base = axis_report(_join(_t, _r0), Y)
table(['변형', 'tie축', '승부축', '해석'],
      [['없음', f4(_base['tie축']), f4(_base['승부축']), '기준'],
       ['t 만 변형', f4(_a['tie축']), f4(_a['승부축']), '승부축 불변이어야 한다'],
       ['r 만 변형', f4(_b['tie축']), f4(_b['승부축']), 'tie축 불변이어야 한다']])

_indep = (abs(_a['승부축'] - _base['승부축']) < 1e-12) and (abs(_b['tie축'] - _base['tie축']) < 1e-12)
# 문턱 1e-6 — 극단 확률(디리클레 0.05)에서 EPS=1e-15 클리핑이 만드는 float64 잔차가 1e-7 급이다.
# 항등식 자체는 대수적으로 정확하고, 여기서 보는 건 구현이 그 정확도를 지키는지다.
verdict(1, '축 분해는 무손실이다', _emax < 1e-6 and _indep,
        f'잔차 최대 {_emax:.1e} (EPS 클리핑 수준), 축 간 누출 없음 '
        '-> 축을 나눠 다루는 것 자체는 비용 0')

## 주장 2 — 두 축은 서로 간섭하지 않는다

주장 1 이 성립하면 $T_\pi$ 는 tie축에만, $T_q$ 는 승부축에만 나타난다.
따라서 **각자 자기 축의 손실만 최소화하면 그게 곧 전체 최소**여야 한다.
말로는 자명하지만 실제로 그런지는 2차원 격자로 직접 훑어 확인한다 — 아니면 이후 절의 전제가 무너진다.

In [ ]:
claim(2, '축별로 따로 맞춘 (T_π, T_q) 가 총 손실의 2차원 공동 최소와 같다')

# 승부축 온도에는 절편을 금지한다 — q(-d) = 1 - q(d) 라는 swap 반대칭이 깨지면 슬롯 편향이 생긴다.
def fit_axis(z, y, bias=False):
    zt = torch.tensor(np.asarray(z, np.float64), dtype=torch.float64)
    yt = torch.tensor(np.asarray(y, np.float64), dtype=torch.float64)
    lg = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    bb = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    prm = [lg, bb] if bias else [lg]
    opt = torch.optim.LBFGS(prm, lr=0.5, max_iter=200, line_search_fn='strong_wolfe')
    def cl():
        opt.zero_grad()
        loss = nn.functional.binary_cross_entropy_with_logits(zt / lg.exp() + bb * bias, yt)
        loss.backward()
        return loss
    opt.step(cl)
    return float(lg.exp().detach()), (float(bb.detach()) if bias else 0.0)

sig = lambda x: 1.0 / (1.0 + np.exp(-x))

def compose(pi, q):
    # (π, q) -> 3-class 확률. 계층 구조 전체가 이 한 줄이다.
    P = np.column_stack([(1 - pi) * q, (1 - pi) * (1 - q), pi])
    return P / P.sum(1, keepdims=True)

_sp, _dq = COL['s_tb'][0], COL['d_abe'][0]
_Ta, _ = fit_axis(_sp, (Y == 2).astype(float))                    # tie축만 보고 적합
_nz = Y != 2
_Tb, _ = fit_axis(_dq[_nz], (Y[_nz] == 0).astype(float))          # 승부축만 보고 적합
_sep = per_row_ll(compose(sig(_sp / _Ta), sig(_dq / _Tb)), Y).mean()

_grid = np.exp(np.linspace(-0.35, 0.35, 71))                      # ±42% 범위를 71x71 로 훑는다
_best, _bll = None, np.inf
for _a in _Ta * _grid:
    for _b in _Tb * _grid:
        _ll = per_row_ll(compose(sig(_sp / _a), sig(_dq / _b)), Y).mean()
        if _ll < _bll:
            _bll, _best = _ll, (_a, _b)
table(['적합 방식', 'T_π', 'T_q', '총 ll'],
      [['축별 (각자 자기 축만)', f'{_Ta:.4f}', f'{_Tb:.4f}', f'{_sep:.6f}'],
       ['2차원 격자 공동최소', f'{_best[0]:.4f}', f'{_best[1]:.4f}', f'{_bll:.6f}']])

# 재정규화 == 로짓차 — q 를 σ(z_a − z_b) 로 써도 되는 근거
_p3 = np.exp(L_ABE - L_ABE.max(1, keepdims=True))
_p3 /= _p3.sum(1, keepdims=True)
_e_renorm = np.abs(_p3[:, 0] / (_p3[:, 0] + _p3[:, 1]) - sig(COL['d_abe'][0])).max()
# swap 반대칭 — 절편을 금지한 이유
_Pf, _Ps = compose(sig(_sp / _Ta), sig(_dq / _Tb)), compose(sig(_sp / _Ta), sig(-_dq / _Tb))
_e_anti = np.abs(_Ps - _Pf[:, [1, 0, 2]]).max()
table(['부수 검산', '오차', '뜻'],
      [['타이 버리고 재정규화 == σ(z_a − z_b)', f'{_e_renorm:.2e}', 'q 를 로짓차로 계산해도 된다'],
       ['P(−d) == P[:, [1,0,2]]', f'{_e_anti:.2e}', '승부축 절편 금지가 반대칭을 보장한다']])

verdict(2, '두 축은 서로 간섭하지 않는다', (_sep - _bll < 1e-6) and _e_renorm < 1e-9 and _e_anti < 1e-12,
        f'축별 {_sep:.6f} vs 격자최소 {_bll:.6f} (차 {_sep - _bll:.1e}) -> 축별 보정이 곧 전체 최적')

## 주장 3 — 각 전문가에게 자기 축만 맡기고 온도 2개면 된다 (방법 1)

$$\pi=\sigma(s_\pi/T_\pi)\ \ \text{(타이 전문가)},\qquad
q_A=\frac{p^{(3)}_A}{p^{(3)}_A+p^{(3)}_B}=\sigma(d/T_q)\ \ \text{(3진 인코더)}$$

학습되는 것은 **스칼라 2개뿐**이다. 부스터도, stage2 학습도 없다.
그래도 자기적합 낙관을 없애려고 온도는 **교차적합**한다 (fold k 의 온도는 나머지 4 fold 로).

축 원천은 손잡이다 — 단일 전문가만 쓸 수도, 같은 축의 로짓을 평균해 쓸 수도 있다.
평균해도 파라미터는 여전히 2개다.

In [ ]:
claim(3, '전문가에게 자기 축만 맡기고 온도 2개만 걸어도 쓸 만한 결합기가 된다')

def run_m1(s_pi, d_q, tie_bias=False):
    pi, q, Tp, Tq, Bp = np.zeros(N), np.zeros(N), [], [], []
    for k in range(N_SPLITS):
        tr, te = FOLD != k, FOLD == k
        a, b = fit_axis(s_pi[tr], (Y[tr] == 2).astype(float), bias=tie_bias)
        nz = tr & (Y != 2)                       # 승부축은 비-tie 행의 조건부 우도로만 적합한다
        c, _ = fit_axis(d_q[nz], (Y[nz] == 0).astype(float), bias=False)
        pi[te], q[te] = sig(s_pi[te] / a + b), sig(d_q[te] / c)
        Tp.append(a); Bp.append(b); Tq.append(c)
    P = compose(pi, q)
    folds = pd.DataFrame([axis_report(P[FOLD == k], Y[FOLD == k]) for k in range(N_SPLITS)])
    return {'oof': P, 'fold': folds, 'mean': folds.mean(), 'std': folds.std(ddof=1),
            'T_pi': Tp, 'T_q': Tq, 'b_pi': Bp}

_mean = lambda *ks: np.mean([COL[k][0] for k in ks], axis=0)
M1_CFG = {
    'M1  π=TB, q=ABE':         (COL['s_tb'][0], COL['d_abe'][0]),          # <- 요청안
    'M1b π=TB, q=ABE+Ari':     (COL['s_tb'][0], _mean('d_abe', *D_ARI)),
    'M1c π=3전문가, q=ABE':     (_mean('s_tb', 's_abe', *S_ARI), COL['d_abe'][0]),
    'M1d π=3전문가, q=ABE+Ari': (_mean('s_tb', 's_abe', *S_ARI), _mean('d_abe', *D_ARI)),
}
REQUESTED = 'M1  π=TB, q=ABE'

for _nm, (_s, _d) in M1_CFG.items():
    RES[_nm] = run_m1(_s, _d)
RES['M1+절편'] = run_m1(*M1_CFG[REQUESTED], tie_bias=True)

def mstd(v):
    return f'{np.mean(v):.3f} ±{np.std(v):.3f}'

table(['구성', '총 ll', '±std', 'tie축', '승부축', 'tie AUC', '승부 AUC', 'T_π', 'T_q'],
      [[nm, f4(RES[nm]['mean']['총 ll']), f4(RES[nm]['std']['총 ll']),
        f4(RES[nm]['mean']['tie축']), f4(RES[nm]['mean']['승부축']),
        f4(RES[nm]['mean']['tie AUC']), f4(RES[nm]['mean']['승부 AUC']),
        mstd(RES[nm]['T_pi']), mstd(RES[nm]['T_q'])]
       for nm in list(M1_CFG) + ['M1+절편']])

_best_m1 = min(list(M1_CFG) + ['M1+절편'], key=lambda k: RES[k]['mean']['총 ll'])
print(f'\n  요청안 {REQUESTED} = {RES[REQUESTED]["mean"]["총 ll"]:.4f}  |  '
      f'변형 최선 {_best_m1} = {RES[_best_m1]["mean"]["총 ll"]:.4f}')
print(f'  T_π/T_q 의 fold 간 표준편차가 0.01 수준이다 — 파라미터 2개짜리라 일반화 분산이 사실상 없다.')
print(f'  T_q < 1 이면 그 조합이 과소확신이라는 뜻이다(로짓평균이 눕힌 것을 온도가 되돌린다).')
print(f'  주의: 변형 4개 중 최선을 고르면 선택 편향이 섞인다. 배포 후보는 사전 지정한 요청안이 기준이다.')

verdict(3, '전문가별 축 분담 + 온도 2개로 결합기가 선다', None,
        f'{REQUESTED} {RES[REQUESTED]["mean"]["총 ll"]:.4f} / 최선 {_best_m1} '
        f'{RES[_best_m1]["mean"]["총 ll"]:.4f} — 학습된 결합기와의 비교는 주장 4')

## 주장 4 — 학습된 joint 부스터가 방법 1 보다 낫다

`J` 는 세 전문가의 **원 로짓**을 그대로 받는 3-class 부스터(현행 배포 형태),
`J-rep` 은 같은 부스터를 **재매개 좌표 `(s, d)`** 로 돌린 것이다.
계층 구성과 좌표를 맞춘 **공정 기준선**이라 이후 판정은 `J-rep` 으로 한다.

stage2 10,748 행으로 학습한 모델이 스칼라 2개를 못 이긴다면, 결합 단계에 남은 신호가 없다는 뜻이다.

In [ ]:
claim(4, '10,748행으로 학습한 joint 부스터가 스칼라 2개짜리 방법 1 보다 낫다')

import inspect
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

def make(name, binary):
    if name == 'lgbm':
        return lgb.LGBMClassifier(objective='binary' if binary else 'multiclass',
                                  **({} if binary else {'num_class': 3}),
                                  learning_rate=0.03, n_estimators=3000, num_leaves=15,
                                  min_child_samples=60, colsample_bytree=0.9, subsample=0.9,
                                  subsample_freq=1, reg_lambda=1.0, random_state=SEED,
                                  n_jobs=-1, verbosity=-1)
    if name == 'xgb':
        return xgb.XGBClassifier(objective='binary:logistic' if binary else 'multi:softprob',
                                 **({} if binary else {'num_class': 3}),
                                 learning_rate=0.03, n_estimators=3000, max_depth=4,
                                 min_child_weight=20, subsample=0.9, colsample_bytree=0.9,
                                 reg_lambda=1.0, tree_method='hist',
                                 eval_metric='logloss' if binary else 'mlogloss',
                                 early_stopping_rounds=100, random_state=SEED,
                                 n_jobs=-1, verbosity=0)
    return cb.CatBoostClassifier(loss_function='Logloss' if binary else 'MultiClass',
                                 eval_metric='Logloss' if binary else 'MultiClass',
                                 iterations=3000, learning_rate=0.03, depth=4, l2_leaf_reg=6.0,
                                 early_stopping_rounds=100, bootstrap_type='Bernoulli',
                                 subsample=0.9, border_count=254, random_seed=SEED,
                                 verbose=0, allow_writing_files=False)

_LGB_NEW = 'eval_X' in inspect.signature(lgb.LGBMClassifier.fit).parameters

def fit_head(name, m, X, y, w, Xe, ye):
    if name == 'lgbm':
        ekw = {'eval_X': Xe, 'eval_y': ye} if _LGB_NEW else {'eval_set': [(Xe, ye)]}
        m.fit(X, y, sample_weight=w, **ekw, callbacks=[lgb.early_stopping(100, verbose=False)])
    elif name == 'xgb':
        m.fit(X, y, sample_weight=w, eval_set=[(Xe, ye)], verbose=False)
    else:
        m.fit(X, y, sample_weight=w, eval_set=(Xe, ye), use_best_model=True)
    return m

def best_it(name, m):
    if name == 'lgbm':
        return int(m.booster_.best_iteration or 0)
    if name == 'xgb':
        return int(getattr(m, 'best_iteration', 0) or 0)
    return int(m.get_best_iteration() or 0)

def cv_joint(name, X, sg_or_perm, is_perm):
    oof = np.zeros((N, 3))
    sw = (lambda A: A[:, sg_or_perm]) if is_perm else (lambda A: A * sg_or_perm)
    swy = lambda y: np.where(y == 0, 1, np.where(y == 1, 0, 2)).astype(y.dtype)
    iters = []
    for k in range(N_SPLITS):
        te, rest = np.where(FOLD == k)[0], np.where(FOLD != k)[0]
        tr, es = train_test_split(rest, test_size=int(round(INNER_FRAC * N)),
                                  stratify=Y[rest], random_state=SEED + k, shuffle=True)
        m = fit_head(name, make(name, False), np.vstack([X[tr], sw(X[tr])]),
                     np.hstack([Y[tr], swy(Y[tr])]), None, X[es], Y[es])
        oof[te] = (m.predict_proba(X[te]) + m.predict_proba(sw(X[te]))[:, [1, 0, 2]]) / 2
        iters.append(best_it(name, m))
    folds = pd.DataFrame([axis_report(oof[FOLD == k], Y[FOLD == k]) for k in range(N_SPLITS)])
    return {'oof': oof, 'fold': folds, 'mean': folds.mean(), 'std': folds.std(ddof=1), 'iters': iters}

if HEAVY:
    _t0 = time.time()
    X_J = np.hstack([L_ABE, L_TB] + ARI_BLOCKS)          # 원 로짓 joint = 현행 배포 형태
    _perm = np.arange(X_J.shape[1])
    _perm[[0, 1]] = [1, 0]
    for _i in range(len(ARI_BLOCKS)):
        _b = 5 + 3 * _i
        _perm[[_b, _b + 1]] = [_b + 1, _b]
    assert (_perm[_perm] == np.arange(len(_perm))).all(), 'J 순열이 involution 이 아니다'
    RES['J 원로짓'] = cv_joint(HEAD_BOOST, X_J, _perm, True)

    _rep = ['s_tb', 's_abe', 'd_abe'] + S_ARI + D_ARI    # 재매개 joint = 공정 기준선
    X_R = np.column_stack([COL[c][0] for c in _rep])
    sg_R = np.array([COL[c][1] for c in _rep])
    RES['J-rep 재매개'] = cv_joint(HEAD_BOOST, X_R, sg_R, False)
    print(f'  부스터 2회 완료 ({time.time() - _t0:.0f}s)')

    _lb = per_row_ll(RES['J-rep 재매개']['oof'], Y)
    table(['구성', '열', '총 ll', '±std', 'tie축', '승부축', 'Δ vs J-rep (양수=이득)'],
          [[nm, {'J 원로짓': X_J.shape[1], 'J-rep 재매개': len(_rep)}.get(nm, 2),
            f4(RES[nm]['mean']['총 ll']), f4(RES[nm]['std']['총 ll']),
            f4(RES[nm]['mean']['tie축']), f4(RES[nm]['mean']['승부축']),
            '기준' if nm == 'J-rep 재매개' else ci_str(*paired_ci(_lb, per_row_ll(RES[nm]['oof'], Y)))]
           for nm in ['J 원로짓', 'J-rep 재매개'] + list(M1_CFG) + ['M1+절편']],
          align='lrrrrrl')
    print(f'\n  재매개 비용 J -> J-rep '
          f'{RES["J-rep 재매개"]["mean"]["총 ll"] - RES["J 원로짓"]["mean"]["총 ll"]:+.4f} '
          '(좌표 효과일 뿐 정보 손실이 아니다. 이후 판정은 J-rep 기준)')

    _d, _lo, _hi = paired_ci(_lb, per_row_ll(RES[_best_m1]['oof'], Y))
    verdict(4, '학습된 joint 부스터가 방법 1 보다 낫다', _lo < 0 and _d < 0,
            f'J-rep {RES["J-rep 재매개"]["mean"]["총 ll"]:.4f} vs {_best_m1} '
            f'{RES[_best_m1]["mean"]["총 ll"]:.4f} — {ci_str(_d, _lo, _hi)}')
else:
    print('  HEAVY=False — 생략')
    verdict(4, '학습된 joint 부스터가 방법 1 보다 낫다', None, 'HEAVY=False 로 미실행')

## 주장 5 — 헤드를 둘로 나눠 **학습**하면 joint 를 이긴다

방법 1 은 축마다 스칼라 하나였다. 여기서는 축마다 **부스터 하나**를 붙인다 —
tie 헤드는 tie 신호들로 `t` 를, AB 헤드는 방향 신호들로 `r` 을 학습한다.

- **tie 헤드**는 입력이 전부 swap 불변이라 **정의상 대칭**이다. 증강도 TTA 도 필요 없다.
  joint 는 증강으로 근사할 뿐이라 절대 정확히 만족하지 못한다.
- **AB 헤드**는 `d` 가 홀함수라 `(x·부호, 1−y)` 증강 + TTA 로 반대칭을 기계 정밀도로 강제한다.
- **AB 학습행**이 쟁점이다. `r` 의 추정 대상은 `P(A│비-tie)` 라 비-tie 행만 쓰는 게 자연스럽지만
  표본을 31% 잃는다. `soft` 는 tie 행을 `y=0.5` 가중 두 벌로 넣어 표본을 joint 와 맞추되
  예측을 0.5 로 당겨 과소확신을 만든다. 무엇이 큰지는 `C0`(soft) vs `C5`(nontie) 가 정한다.

| 구성 | tie 헤드 | AB 헤드 | 학습행 | 무엇을 가르나 |
|---|---|---|---|---|
| `C0` 요청안 | `s_tb, s_abe, s_ari` | `notie_raw, d_abe, d_ari` | soft | 헤드라인 |
| `C1` +마진 | `C0 + |d|` | 〃 | tie 헤드에 마진이 필요한가 |
| `C2` TB만 | `s_tb` | 〃 | ABE/Arith 의 tie 신호가 잉여인가 |
| `C3` TB빼기 | `C0` | `d_abe, d_ari` | 〃 | `notie` 가 AB 에 값을 하나 (축간 상호작용) |
| `C4` 정규화notie | `C0` | `notie_s, ...` | 〃 | raw 스케일 vs 정규화 |
| `C5` 비-tie학습 | `C0` | `C0` | nontie | soft 의 과소확신 vs 표본 손실 |
| `C6` 최선조합 | `C1` | `C0` | nontie | 위에서 이긴 것들만 합친 판 |

In [ ]:
claim(5, '헤드를 둘로 나눠 학습하면 joint 부스터를 이긴다')

# AB 헤드 학습 표본. nontie: 대상과 정확히 일치하나 표본 31% 손실 / soft: 표본은 같으나 0.5 로 당김
def win_rows(idx, XW, win_train):
    nz = idx[Y[idx] != 2]
    Xs, ys, ws = [XW[nz]], [(Y[nz] == 0).astype(np.int64)], [np.ones(len(nz))]
    if win_train == 'soft':
        ti = idx[Y[idx] == 2]
        Xs += [XW[ti], XW[ti]]
        ys += [np.ones(len(ti), np.int64), np.zeros(len(ti), np.int64)]
        ws += [np.full(len(ti), 0.5), np.full(len(ti), 0.5)]
    return np.vstack(Xs), np.hstack(ys), np.hstack(ws)

def cv_cascade(name, tie_names, ab_names, win_train=None, verify=False):
    win_train = WIN_TRAIN if win_train is None else win_train
    XT, sgT = build(tie_names, 'tie')
    XW, sgW = build(ab_names, 'ab')
    assert np.array_equal(XT, XT * sgT), 'tie 헤드 입력이 swap 불변이 아니다'
    yt = (Y == 2).astype(np.int64)
    t_oof, r_oof, iters, anti = np.zeros(N), np.zeros(N), [], 0.0
    for k in range(N_SPLITS):
        te, rest = np.where(FOLD == k)[0], np.where(FOLD != k)[0]

        # tie 헤드 — 구조적 대칭이라 증강·TTA 불필요
        tr, es = train_test_split(rest, test_size=int(round(INNER_FRAC * N)),
                                  stratify=yt[rest], random_state=SEED + k, shuffle=True)
        m1 = fit_head(name, make(name, True), XT[tr], yt[tr], None, XT[es], yt[es])
        t_oof[te] = m1.predict_proba(XT[te])[:, 1]

        # AB 헤드 — 부호반전 증강으로 반대칭 강제. ES 는 비-tie 행(= 승부축 지표)으로만
        nz = rest[Y[rest] != 2]
        _, es2 = train_test_split(nz, test_size=max(50, int(round(INNER_FRAC * len(nz)))),
                                  stratify=(Y[nz] == 0).astype(int), random_state=SEED + k, shuffle=True)
        keep = np.setdiff1d(rest, es2)
        Xw, yw, ww = win_rows(keep, XW, win_train)
        m2 = fit_head(name, make(name, True), np.vstack([Xw, Xw * sgW]),
                      np.hstack([yw, 1 - yw]).astype(np.int64), np.hstack([ww, ww]),
                      XW[es2], (Y[es2] == 0).astype(np.int64))
        _f = m2.predict_proba(XW[te])[:, 1]
        _s = m2.predict_proba(XW[te] * sgW)[:, 1]
        r_oof[te] = (_f + 1 - _s) / 2
        anti = max(anti, float(np.abs((_f + 1 - _s) / 2 + (_s + 1 - _f) / 2 - 1).max()))
        iters.append((best_it(name, m1), best_it(name, m2)))
    assert anti < 1e-12, f'AB 헤드 반대칭 위반 {anti:.2e}'
    p = compose(t_oof, r_oof)
    folds = pd.DataFrame([axis_report(p[FOLD == k], Y[FOLD == k]) for k in range(N_SPLITS)])
    return {'oof': p, 'fold': folds, 'mean': folds.mean(), 'std': folds.std(ddof=1),
            'iters': iters, 'anti': anti}

TIE0 = ['s_tb', 's_abe'] + S_ARI
TIE1 = TIE0 + ['|d_abe|'] + AD_ARI
AB0  = ['notie_raw', 'd_abe'] + D_ARI
CONFIGS = {
    'C0 요청안':      (TIE0, AB0, 'soft'),
    'C1 +마진':       (TIE1, AB0, 'soft'),
    'C2 TB만':        (['s_tb'], AB0, 'soft'),
    'C3 TB빼기':      (TIE0, ['d_abe'] + D_ARI, 'soft'),
    'C4 정규화notie': (TIE0, ['notie_s', 'd_abe'] + D_ARI, 'soft'),
    'C5 비-tie학습':  (TIE0, AB0, 'nontie'),
    'C6 최선조합':    (TIE1, AB0, 'nontie'),
}

if HEAVY:
    _t0 = time.time()
    for nm, (tn, an, wt) in CONFIGS.items():
        RES[nm] = cv_cascade(HEAD_BOOST, tn, an, wt)
    print(f'  부스터 {len(CONFIGS) * 2}회 완료 ({time.time() - _t0:.0f}s) | '
          f'AB 헤드 반대칭 오차 최대 {max(RES[c]["anti"] for c in CONFIGS):.2e} '
          '| tie 헤드 swap 불변 (행렬 동일) ✓')

    _lb = per_row_ll(RES['J-rep 재매개']['oof'], Y)
    table(['구성', '총 ll', '±std', 'tie축', '승부축', 'tie AUC', '승부 AUC', 'Δ vs J-rep (양수=이득)'],
          [[nm, f4(RES[nm]['mean']['총 ll']), f4(RES[nm]['std']['총 ll']),
            f4(RES[nm]['mean']['tie축']), f4(RES[nm]['mean']['승부축']),
            f4(RES[nm]['mean']['tie AUC']), f4(RES[nm]['mean']['승부 AUC']),
            '기준' if nm == 'J-rep 재매개' else ci_str(*paired_ci(_lb, per_row_ll(RES[nm]['oof'], Y)))]
           for nm in ['J 원로짓', 'J-rep 재매개', _best_m1] + list(CONFIGS)], align='lrrrrrrl')

    print()
    table(['대비 (양수 = 뒤쪽이 이득)', '결과', '무엇을 가르나'],
          [[f'{a} -> {b}', ci_str(*paired_ci(per_row_ll(RES[a]['oof'], Y), per_row_ll(RES[b]['oof'], Y))), q]
           for a, b, q in (
               ('C0 요청안', 'C1 +마진', 'tie 헤드에 마진(|d|)이 필요한가'),
               ('C2 TB만', 'C0 요청안', 'ABE/Arith tie 신호가 TB 위에 값을 더하나'),
               ('C3 TB빼기', 'C0 요청안', 'notie 가 AB 헤드에 값을 하나 (축간 상호작용 크기)'),
               ('C0 요청안', 'C4 정규화notie', 'raw 스케일 vs 정규화'),
               ('C0 요청안', 'C5 비-tie학습', 'soft 의 과소확신 vs 표본 31% 손실'),
               ('C5 비-tie학습', 'C6 최선조합', '그 위에 마진을 더하면'),
               ('J-rep 재매개', 'C6 최선조합', '최선 계층 구성이 joint 를 이기나'))], align='llr')

    print('\n[fold 별 총 ll]')
    table(['구성'] + [f'fold{k}' for k in range(N_SPLITS)],
          [[nm] + [f4(v) for v in RES[nm]['fold']['총 ll'].values]
           for nm in ['J-rep 재매개', _best_m1] + list(CONFIGS)])

    _cbest = min(CONFIGS, key=lambda k: RES[k]['mean']['총 ll'])
    _d, _lo, _hi = paired_ci(_lb, per_row_ll(RES[_cbest]['oof'], Y))
    verdict(5, '헤드를 나눠 학습하면 joint 를 이긴다', _lo > 0 and _d >= ADOPT_THRESH,
            f'최선 {_cbest} {RES[_cbest]["mean"]["총 ll"]:.4f} vs J-rep '
            f'{RES["J-rep 재매개"]["mean"]["총 ll"]:.4f} — {ci_str(_d, _lo, _hi)}')
else:
    print('  HEAVY=False — 생략')
    verdict(5, '헤드를 나눠 학습하면 joint 를 이긴다', None, 'HEAVY=False 로 미실행')

## 주장 6 — 그래서 배포에 계층 구조를 넣어야 한다

**동률이면 채택하지 않는다.** tie 헤드의 구조적 대칭으로 아끼는 것은 부스터 예측 한 번뿐이고,
비용의 사실상 전부인 인코더 forward 는 그대로다. 부수 이득이 없으므로 **총점이 문턱(0.0005)을
넘겨 이길 때만** 계층화를 채택한다.

배포 구성도 CV argmin 으로 고르지 않는다 — 보고 지표와 같은 CV 로 고르면 낙관이 섞인다.
사전 지정안을 기준으로 두고, paired bootstrap 이 유의하고 문턱을 넘은 후보만 그것을 대체한다.

In [ ]:
claim(6, '배포에 계층 구조를 넣어야 한다 (동률이면 채택하지 않는다)')

if HEAVY:
    _cand = min(CONFIGS, key=lambda k: RES[k]['mean']['총 ll'])
    _d, _lo, _hi = paired_ci(per_row_ll(RES[DEFAULT_CFG]['oof'], Y), per_row_ll(RES[_cand]['oof'], Y))
    if DEPLOY_CFG is not None:
        BEST, _why = DEPLOY_CFG, '손잡이 강제'
    elif _cand != DEFAULT_CFG and _lo > 0 and _d >= ADOPT_THRESH:
        BEST, _why = _cand, f'문턱 {ADOPT_THRESH} 초과 + CI 유의'
    else:
        BEST, _why = DEFAULT_CFG, '문턱 미달 — 사전 지정안 유지'
    assert BEST in CONFIGS, f'알 수 없는 구성: {BEST}'

    _jr = RES['J-rep 재매개']['mean']['총 ll']
    _dj, _loj, _hij = paired_ci(per_row_ll(RES['J-rep 재매개']['oof'], Y),
                                per_row_ll(RES[BEST]['oof'], Y))
    table(['항목', '값'],
          [['사전 지정안', f'{DEFAULT_CFG}  ({RES[DEFAULT_CFG]["mean"]["총 ll"]:.4f})'],
           ['CV argmin', f'{_cand}  ({RES[_cand]["mean"]["총 ll"]:.4f})'],
           ['사전안 -> argmin', ci_str(_d, _lo, _hi)],
           ['선택된 배포 구성', f'{BEST}  ({_why})'],
           ['joint(J-rep) 대비', ci_str(_dj, _loj, _hij)],
           ['방법 1 최선 대비', ci_str(*paired_ci(per_row_ll(RES[_best_m1]['oof'], Y),
                                                per_row_ll(RES[BEST]['oof'], Y)))]], align='ll')
    if BEST == _cand and _cand != DEFAULT_CFG:
        print(f'\n  주의: {BEST} 의 CV 평균은 같은 CV 로 고른 값이라 낙관이다. 무편향 값에는 새 데이터가 필요하다.')

    verdict(6, '배포에 계층 구조를 넣어야 한다', _loj > 0 and _dj >= ADOPT_THRESH,
            f'{BEST} {RES[BEST]["mean"]["총 ll"]:.4f} vs J-rep {_jr:.4f} — {ci_str(_dj, _loj, _hij)}'
            + ('' if (_loj > 0 and _dj >= ADOPT_THRESH) else ' -> 동률/열세라 joint 유지'))
else:
    BEST = None
    print('  HEAVY=False — 생략')
    verdict(6, '배포에 계층 구조를 넣어야 한다', None, 'HEAVY=False 로 미실행')

## 주장 7 — 이기든 지든 **어느 층에서** 갈렸는지 설명된다

총 ll 은 평균이라 "어느 행이 얼마를 물어냈는지"를 감춘다. `J-rep` 대비 **이동만** 한 표로 본다.
특히 `5) 층화` 가 중요하다 — 계층 구조는 tie 판정을 전용 헤드에 맡겼으므로,
**두 전문가가 엇갈리는 층(`ABE만 tie` / `TB만 tie`)에서 joint 보다 나아야** 구조가 값을 한 것이다.
`둘 다 비-tie`(83%)에서만 움직였다면 그건 구조가 아니라 캘리브레이션 차이다.

In [ ]:
claim(7, '총점 차이가 어느 층에서 나왔는지 설명된다')

_ap = np.exp(L_ABE - L_ABE.max(1, keepdims=True)); _ap /= _ap.sum(1, keepdims=True)
_tp = np.exp(L_TB - L_TB.max(1, keepdims=True)); _tp /= _tp.sum(1, keepdims=True)
ABE_TIE, TB_TIE = _ap[:, 2] > 0.5, _tp[:, 1] > 0.5
STRATA = (('둘 다 tie', ABE_TIE & TB_TIE), ('둘 다 비-tie', ~ABE_TIE & ~TB_TIE),
          ('ABE만 tie', ABE_TIE & ~TB_TIE), ('TB만 tie', ~ABE_TIE & TB_TIE))
CLS = ('A승', 'B승', 'tie')

def dissect(p, y):
    e = per_row_ll(p, y)
    rep = axis_report(p, y)
    conf, hit = p.max(1), (p.argmax(1) == y).astype(float)
    b = np.clip((conf * 10).astype(int), 0, 9)
    ece = sum(np.mean(b == i) * abs(conf[b == i].mean() - hit[b == i].mean())
              for i in range(10) if (b == i).sum() >= 20)
    o = np.argsort(-e)
    out = {'총 ll': float(e.mean()), 'tie축': rep['tie축'], '승부축': rep['승부축'],
           'ECE': float(ece), '확신−정답': float(conf.mean() - hit.mean())}
    for q in (0.05, 0.20):
        out[f'상위{int(q * 100)}% 손실비중'] = float(e[o[:int(q * len(y))]].sum() / e.sum() * 100)
    for c, nm in enumerate(CLS):
        out[f'{nm} 평균손실'] = float(e[y == c].mean())
    for nm, m in STRATA:
        out[f'{nm} 손실'] = float(e[m].mean()) if m.sum() >= 30 else np.nan
    return out

REF = 'J-rep 재매개' if HEAVY else _best_m1
_targets = [n for n in dict.fromkeys(([REF] + ([_best_m1] if HEAVY else list(M1_CFG))
                                      + ([DEFAULT_CFG, BEST] if HEAVY else []))) if n in RES]
D = {nm: dissect(RES[nm]['oof'], Y) for nm in _targets}
_keys = list(D[REF])
_others = [n for n in _targets if n != REF]
table([f'항목 (기준 {REF})', REF] + [f'Δ {n}' for n in _others],
      [[k, f'{D[REF][k]:.4f}'] + [f'{D[n][k] - D[REF][k]:+.4f}' for n in _others] for k in _keys])
print('\n  층화 n: ' + ' | '.join(f'{nm} {int(m.sum()):,}' for nm, m in STRATA))
print('  · 음수 = 그 구성이 그 층에서 이득. 엇갈리는 층에서 음수여야 구조가 중재로 값을 한 것이다')
print('  · tie축이 내렸는데 승부축이 오르면(또는 반대면) 구조가 축을 옮긴 것이지 번 게 아니다')

# 층화는 전 행을 남김없이 쪼갠다 -> 총점 차이 = Σ (n_s/N)·(층 손실 차) 가 **항등식**이어야 한다.
# 이게 성립해야 "차이의 출처를 층으로 설명했다"고 말할 수 있다(설명이 아니라 재구성이다).
if _others:
    _tgt = _others[0]
    _cov = sum(int(m.sum()) for _, m in STRATA)
    _recon = sum((m.sum() / N) * (D[_tgt][f'{nm} 손실'] - D[REF][f'{nm} 손실']) for nm, m in STRATA)
    _tot = D[_tgt]['총 ll'] - D[REF]['총 ll']
    _share = sorted(((nm, (m.sum() / N) * (D[_tgt][f'{nm} 손실'] - D[REF][f'{nm} 손실']))
                     for nm, m in STRATA), key=lambda kv: -abs(kv[1]))
    table(['재구성 검산', '값'],
          [['층화가 덮는 행', f'{_cov:,} / {N:,}'],
           [f'총점 차이 ({_tgt} − {REF})', f'{_tot:+.6f}'],
           ['층별 기여 합', f'{_recon:+.6f}'],
           ['최대 기여 층', f'{_share[0][0]} {_share[0][1]:+.4f} ({_share[0][1] / _tot * 100:.0f}%)'
            if abs(_tot) > 1e-12 else '-']], align='ll')
    verdict(7, '총점 차이의 출처가 층 단위로 설명된다',
            _cov == N and abs(_recon - _tot) < 1e-9,
            f'층별 기여 합 {_recon:+.6f} == 총점 차이 {_tot:+.6f}, '
            f'최대 기여 {_share[0][0]} ({_share[0][1] / _tot * 100:.0f}%)'
            if abs(_tot) > 1e-12 else f'층별 기여 합 {_recon:+.6f} == 총점 차이 {_tot:+.6f}')
else:
    verdict(7, '총점 차이의 출처가 층 단위로 설명된다', None, '비교 대상 부족')

### §8 배포 저장 — 선택된 구성으로 stage2 전량 재적합

In [ ]:
if HEAVY and BEST is not None:
    tie_names, ab_names, best_wt = CONFIGS[BEST]
    XT, sgT = build(tie_names, 'tie')
    XW, sgW = build(ab_names, 'ab')
    yt = (Y == 2).astype(np.int64)
    tr, es = train_test_split(np.arange(N), test_size=int(round(INNER_FRAC * N)),
                              stratify=Y, random_state=SEED, shuffle=True)
    M_TIE = fit_head(HEAD_BOOST, make(HEAD_BOOST, True), XT[tr], yt[tr], None, XT[es], yt[es])
    nz = np.arange(N)[Y != 2]
    _, es2 = train_test_split(nz, test_size=max(50, int(round(INNER_FRAC * len(nz)))),
                              stratify=(Y[nz] == 0).astype(int), random_state=SEED, shuffle=True)
    Xw, yw, ww = win_rows(np.setdiff1d(np.arange(N), es2), XW, best_wt)
    M_AB = fit_head(HEAD_BOOST, make(HEAD_BOOST, True), np.vstack([Xw, Xw * sgW]),
                    np.hstack([yw, 1 - yw]), np.hstack([ww, ww]),
                    XW[es2], (Y[es2] == 0).astype(np.int64))

    _rows = []
    for nm, m, cv_i in (('tie', M_TIE, [i[0] for i in RES[BEST]['iters']]),
                        ('ab', M_AB, [i[1] for i in RES[BEST]['iters']])):
        bi, med = best_it(HEAD_BOOST, m), int(np.median(cv_i))
        _rows.append([nm, bi, med, '경고: 2배 이상 차이' if bi and (bi > 2 * med or bi * 2 < med) else ''])
        d = MODEL_DIR / nm
        os.makedirs(d, exist_ok=True)
        (m.booster_.save_model(str(d / 'model.txt')) if HEAD_BOOST == 'lgbm'
         else m.save_model(str(d / ('model.json' if HEAD_BOOST == 'xgb' else 'model.cbm'))))
    table(['헤드', 'best_iter', 'CV 중앙값', ''], _rows, align='lrrl')

    meta = {
        'created': time.strftime('%Y-%m-%d %H:%M'),
        'structure': 'cascade: t=tie head, r=AB head, p=[(1-t)r, (1-t)(1-r), t]',
        'booster': HEAD_BOOST, 'best_config': BEST, 'default_config': DEFAULT_CFG,
        'adopt_thresh': ADOPT_THRESH, 'stage2_rows': N, 'n_splits': N_SPLITS,
        'inner_frac': INNER_FRAC, 'win_train': best_wt,
        'arith': {'src': ARI_PRE, 'head': ARITH_HEAD, 'models': list(ARITH_MODELS)},
        # 열 순서와 swap 부호가 배포 계약이다 — 어긋나면 조용히 틀린 답을 낸다
        'tie_head': {'columns': tie_names, 'swap_sign': sgT.tolist(),
                     'note': '전 열 swap 불변 -> 증강·TTA 불필요'},
        'ab_head': {'columns': ab_names, 'swap_sign': sgW.tolist(),
                    'note': 'r = (f(x) + 1 - f(x*sign))/2 로 TTA. 반대칭이 기계 정밀도로 성립'},
        'experts': {'abe': {'steps': ABE_STEPS, 'root': str(ABE_ROOT)},
                    'tb': {'steps': TB_STEPS, 'root': str(TB_ROOT),
                           'note': 'swap 불변 — [1,0,2] 재정렬 금지'}},
        'method1': {nm: {'T_pi': RES[nm]['T_pi'], 'T_q': RES[nm]['T_q']} for nm in M1_CFG},
        'cv_mean': {k: {kk: float(vv) for kk, vv in v['mean'].items()} for k, v in RES.items()},
        'cv_std_ll': {k: float(v['std']['총 ll']) for k, v in RES.items()},
        'verdicts': VERDICTS,
    }
    with open(MODEL_DIR / 'cascade_meta.json', 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    print(f'\nsaved: {MODEL_DIR}/{{tie,ab}}/  |  {MODEL_DIR}/cascade_meta.json')

np.savez_compressed(OOF_PATH, ids=IDS, y=Y,
                    **{f'oof__{k.split()[0]}': v['oof'].astype(np.float32) for k, v in RES.items()})
print(f'saved: {OOF_PATH}  ({len(RES)}개 구성의 OOF)')

### §9 판정 요약 — 여기만 봐도 된다

In [ ]:
table(['#', '주장', '판정', '근거'],
      [[v['#'], v['주장'], v['판정'], v['근거']] for v in VERDICTS], align='rlll')

_all = {k: v['mean']['총 ll'] for k, v in RES.items()}
print()
_npar = lambda k: ('스칼라 3' if k == 'M1+절편' else '스칼라 2' if k.startswith('M1')
                   else '부스터 1' if k.startswith('J') else '부스터 2')
table(['순위', '구성', '총 ll', 'tie축', '승부축', '학습 파라미터'],
      [[i, k, f4(v), f4(RES[k]['mean']['tie축']), f4(RES[k]['mean']['승부축']), _npar(k)]
       for i, (k, v) in enumerate(sorted(_all.items(), key=lambda kv: kv[1]), 1)], align='rlrrrl')
print('\n  같은 fold 분할이라 Boost_Ens.ipynb 의 ATR·cat 5-fold 평균과 직접 비교된다.')